In [1]:
# View and modify the working path
import os
from google.colab import drive

# View current working directory
print("Current Working Directory:", os.getcwd())

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
path = "/content/gdrive/My Drive/BD4H/data"
os.chdir(path)

# Confirm the change
print("Working Directory:", os.getcwd())

Current Working Directory: /content
Mounted at /content/gdrive
Working Directory: /content/gdrive/My Drive/BD4H/data


In [ ]:
#### HuggingFace Token
HF_TOKEN = 'xxxxxxxxxxxxxxxx'


In [3]:
import pandas as pd

readmission_positive = pd.read_csv('/content/gdrive/My Drive/BD4H/data/readmission_positive.csv')
readmission_negative = pd.read_csv('/content/gdrive/My Drive/BD4H/data/readmission_negative.csv')
readmission_30d_positive = pd.read_csv('/content/gdrive/My Drive/BD4H/data/readmission_30d_positive.csv')
readmission_30d_negative = pd.read_csv('/content/gdrive/My Drive/BD4H/data/readmission_30d_negative.csv')

# pd.set_option('display.max_colwidth', None)
print(readmission_positive.shape, readmission_negative.shape)
display(readmission_positive.head(3))
display(readmission_negative.head(3))
print('\n')
print(readmission_30d_positive.shape, readmission_30d_negative.shape)
display(readmission_30d_positive.head(3))
display(readmission_30d_negative.head(3))

(3544, 3) (3544, 3)


,HADM_ID,cleaned_text,gen_readm
0,100018.0,Admission Date: [**--**] Discharge Date: [**--...,positive
1,100020.0,Admission Date: [**--**] Discharge Date: [**--...,positive
2,100039.0,Admission Date: [**--**] Discharge Date: [**--...,positive


,HADM_ID,cleaned_text,gen_readm
0,165403.0,Admission Date: [**--**] Discharge Date: [**--...,negative
1,155222.0,Admission Date: [**--**] Discharge Date: [**--...,negative
2,162999.0,Admission Date: [**--**] Discharge Date: [**--...,negative




(963, 3) (963, 3)


,HADM_ID,cleaned_text,readm_30d
0,100020.0,Admission Date: [**--**] Discharge Date: [**--...,positive
1,100039.0,Admission Date: [**--**] Discharge Date: [**--...,positive
2,100120.0,Admission Date: [**--**] Discharge Date: [**--...,positive


,HADM_ID,cleaned_text,readm_30d
0,195991.0,Admission Date: [**--**] Discharge Date: [**--...,negative
1,192808.0,Admission Date: [**--**] Discharge Date: [**--...,negative
2,133926.0,Admission Date: [**--**] Discharge Date: [**--...,negative


In [4]:
%%capture
!pip install transformers evaluate accelerate

In [5]:
from transformers import AutoTokenizer, DataCollatorWithPadding

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

import evaluate
import numpy as np
from datasets import Dataset, DatasetDict
import pandas as pd
from sklearn.model_selection import train_test_split

In [6]:
# Create readmission and readmission datasets
readmission = pd.concat([readmission_positive, readmission_negative])
readmission_30d = pd.concat([readmission_30d_positive, readmission_30d_negative])

# Rename columns
readmission = readmission.rename(columns={'cleaned_text': 'text', 'gen_readm': 'label'})
readmission_30d = readmission_30d.rename(columns={'cleaned_text': 'text', 'readm_30d': 'label'})

label_mapping = {'negative': 0, 'positive': 1}
readmission['label'] = readmission['label'].map(label_mapping)
readmission_30d['label'] = readmission_30d['label'].map(label_mapping)

# Split the data into train and test sets
train_readmission, test_readmission = train_test_split(readmission, test_size=0.3, random_state=42)
train_readmission_30d, test_readmission_30d = train_test_split(readmission_30d, test_size=0.3, random_state=42)

# Create DatasetDict
readmission_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_readmission),
    'test': Dataset.from_pandas(test_readmission)
})

readmission_30d_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_readmission_30d),
    'test': Dataset.from_pandas(test_readmission_30d)
})

# DistillBERT

In [13]:
# Set up tokenizer and model

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_readmission = readmission_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
)



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4961 [00:00<?, ? examples/s]

Map:   0%|          | 0/2127 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Define metrics for model training

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [16]:
# Start to traing bio-clinical fine-tuned BERT

import os
import wandb

os.environ["WANDB_MODE"] = "disabled"
wandb.init(mode="disabled")

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    push_to_hub=False,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)

model.config.hidden_dropout_prob = 0.4
model.config.attention_probs_dropout_prob = 0.4

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_readmission["train"],
    eval_dataset=tokenized_readmission["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.665497,0.586742,0.644833,0.410946,0.501983
2,0.674300,0.649752,0.619652,0.637666,0.577922,0.606326
3,0.674300,0.673895,0.608369,0.688752,0.414657,0.517661
4,0.634000,0.664263,0.631876,0.633726,0.648423,0.640990


TrainOutput(global_step=1244, training_loss=0.6473905679880615, metrics={'train_runtime': 1120.5034, 'train_samples_per_second': 22.137, 'train_steps_per_second': 1.388, 'total_flos': 2628683058929664.0, 'train_loss': 0.6473905679880615, 'epoch': 4.0})

In [17]:
# Readmission-30d
tokenized_readmission_30d = readmission_30d_dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    push_to_hub=False,
    learning_rate=2e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)

model.config.hidden_dropout_prob = 0.2
model.config.attention_probs_dropout_prob = 0.2

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_readmission_30d["train"],
    eval_dataset=tokenized_readmission_30d["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

Map:   0%|          | 0/578 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.664820,0.622837,0.612040,0.642105,0.626712
2,No log,0.664351,0.626298,0.620209,0.624561,0.622378
3,No log,0.664710,0.619377,0.607261,0.645614,0.625850
4,No log,0.664434,0.624567,0.616438,0.631579,0.623917


TrainOutput(global_step=340, training_loss=0.6707813038545496, metrics={'train_runtime': 320.7404, 'train_samples_per_second': 21.014, 'train_steps_per_second': 1.325, 'total_flos': 714264213553152.0, 'train_loss': 0.6707813038545496, 'epoch': 4.0})

# ClinicalBERT

In [7]:
# Set up tokenizer and model

# tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased") # A distilled bert model can only achieve ~60% accuracy. Not very promising
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT") # Turned to a bio-clinical fine-tuned bert model, trained with mimic-iii data

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512) # Add max_length to fit the bio-clinical BERT, otherwise, it's not needed

tokenized_readmission = readmission_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id
# )

model = AutoModelForSequenceClassification.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT", num_labels=2, id2label=id2label, label2id=label2id
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Map:   0%|          | 0/4961 [00:00<?, ? examples/s]

Map:   0%|          | 0/2127 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from huggingface_hub import login
login()

In [9]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Define metrics for model training

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [12]:
# Start to traing bio-clinical fine-tuned BERT

import os
import wandb

os.environ["WANDB_MODE"] = "disabled"
wandb.init(mode="disabled")

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    push_to_hub=False,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)

model.config.hidden_dropout_prob = 0.5
model.config.attention_probs_dropout_prob = 0.5

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_readmission["train"],
    eval_dataset=tokenized_readmission["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.681483,0.562294,0.540585,0.908163,0.677743
2,0.666700,0.647897,0.625294,0.669073,0.515770,0.582504
3,0.666700,0.703431,0.628115,0.660694,0.547310,0.598681
4,0.561700,0.749568,0.628585,0.637405,0.619666,0.628410


TrainOutput(global_step=1244, training_loss=0.5735927778041631, metrics={'train_runtime': 2204.3579, 'train_samples_per_second': 11.253, 'train_steps_per_second': 0.705, 'total_flos': 5221175782563840.0, 'train_loss': 0.5735927778041631, 'epoch': 4.0})

In [ ]:
# Readmission-30d
tokenized_readmission_30d = readmission_30d_dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    push_to_hub=False,
    learning_rate=2e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    lr_scheduler_type="linear",
    warmup_steps=100,
)

model.config.hidden_dropout_prob = 0.2
model.config.attention_probs_dropout_prob = 0.2

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_readmission_30d["train"],
    eval_dataset=tokenized_readmission_30d["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

Map:   0%|          | 0/578 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.149834,0.707612,0.660221,0.838596,0.738794
2,No log,0.649381,0.695502,0.681063,0.719298,0.699659
3,No log,0.634475,0.705882,0.687296,0.740351,0.712838
4,No log,0.635162,0.700692,0.679487,0.743860,0.710218
5,No log,0.633212,0.700692,0.685430,0.726316,0.705281


TrainOutput(global_step=425, training_loss=0.7531679400275735, metrics={'train_runtime': 743.0846, 'train_samples_per_second': 9.07, 'train_steps_per_second': 0.572, 'total_flos': 1773368513126400.0, 'train_loss': 0.7531679400275735, 'epoch': 5.0})